<a href="https://colab.research.google.com/github/dhruvi003/ai-engineering-learning/blob/main/chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# setup

In [3]:
!pip install openai anthropic tiktoken -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 469.4/469.4 kB 8.6 MB/s eta 0:00:00


In [4]:
from openai import OpenAI
import tiktoken
import json
import anthropic

In [5]:
from datetime import datetime

In [6]:
from typing import Optional

In [7]:
GROQ_CLIENT = OpenAI(
    api_key = "...",
    base_url = "https://api.groq.com/openai/v1"
)

In [8]:
ANTRHOPIC_CLIENT = anthropic.Anthropic(
    api_key = "...",
    # base_url = "https://api.groq.com/anthropic/v1"
)

In [9]:
MODELS = {
    "groq": "llama-3.1-8b-instant",
    "anthropic": "claude-haiku-4-5-20251001"
}

In [ ]:
## Defining Tools

In [10]:
def get_weather(city: str)-> str:
    ans = f'weather for {city} is 35 degree celcius'
    return ans

In [11]:
def calculate(expression: str)-> str:
    try:
        ans = eval(expression)
        return ans
    except Exception as e:
        return e

In [12]:
def get_current_datetime() -> str:
    time_now = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
    return time_now

In [ ]:
datetime.now().strftime('%Y-%m-%d %H:%M:%S')

'2026-03-25 08:05:37'

In [2]:
## defining tool schema
TOOL_SCHEMA = [
   {
       "type":"function",
       "function":{
           "name": "get_weather_data",
           "description": "This function will give weather data",
           "parameters": {
               "type": "object",
               "properties": {
                   "city":{
                       "type": "string",
                       "description": "name of the city"
                   }
               },
               "required": ["city"]
           }
       }
   },
    {
        "type": "function",
        "function": {
        "name": "calculate",
        "description": "Calculate the result of a given expression",
        "parameters": {
            "type":"object",
            "properties":{
                "expression":{
                    "type": "string",
                    "description": "expression to be evaluated"
                }
            },
            "required": ["expression"]
        },
      }
    },
     {
        "type": "function",
        "function" : {
            "name": "get_current_datetime",
            "description": "This function will give current time",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]


In [13]:
## defining tool registry

TOOLS = {
    "get_weather_data": get_weather,
    "calculate": calculate,
    "get_current_datetime": get_current_datetime

}

In [ ]:
## Chatbot class


In [14]:
class ChatBot:
    def __init__(self,
                 provider: str = "groq",
                 system_prompt:str="You are a helpful assistant",
                 max_token_before_trim:int=300
                 ):

          self.provider = provider
          self.system_prompt = system_prompt
          self.max_token = max_token_before_trim
          self.history = []
          self.total_input_tokens = 0
          self.total_output_tokens = 0


    def _call_llm(self, message: list) -> object:
          if self.provider.lower() == "groq":
              res = GROQ_CLIENT.chat.completions.create(
                  model = MODELS["groq"],
                  messages = message,
                  tools = TOOL_SCHEMA,
                  tool_choice="auto"

              )
          elif self.provider.lower() == "anthropic":
              res = ANTRHOPIC_CLIENT.messages.create(
                  model = MODELS["anthropic"],
                  messages = message,
                  system = self.system_prompt,
                  max_tokens=self.max_token
              )
          return res

    def _extract_reply_and_tool_calls(self, response) -> tuple:
          if self.provider == 'groq':
              llm_reply = response.choices[0].message.content
              tool_calling_name = response.choices[0].message.tool_calls

              if tool_calling_name == None:
                  return (llm_reply, None)
              else:
                  return (llm_reply, tool_calling_name)
          else:
              print("dumbass, just use groq for now")

    def _update_token_count( self, response):
        if self.provider == "groq":
            self.total_input_tokens += response.usage.prompt_tokens
            self.total_output_tokens += response.usage.completion_tokens
            return

    def _trim_if_needed(self):
        while len(str(self.history))//4 > self.max_token:
            self.history = self.history[2:] # removing oldest 2 messages

    def chat(self, message:str )-> str:
        self.history.append({"role": "user", "content": message})
        self._trim_if_needed()
        res = self._call_llm(self.history)
        self._update_token_count(res)
        # call tool
        reply, tool_calls = self._extract_reply_and_tool_calls(res)

        if tool_calls is not None:
            self.history.append(res.choices[0].message)

            for tc in tool_calls:
                tool_name = tc.function.name
                tool_args = json.loads(tc.function.arguments)

                tool_fn = TOOLS[tool_name]
                tool_result = tool_fn(**tool_args)

                self.history.append({
                    "role":"tool",
                    "tool_call_id": tc.id,
                    "content": tool_result
                })

            res2 = self._call_llm(self.history)
            self._update_token_count(res2)
            reply = res2.choices[0].message.content
        self.history.append({"role": "assistant", "content": reply})
        return reply

    def help(self) -> str:
        tools_list = "\n".join([
            f"  - {t['function']['name']}: {t['function']['description']}"
            for t in TOOL_SCHEMA
        ])
        return f"Available tools:\n{tools_list}\n\nCommands: /help  /history  /cost  /quit"

    def history_summary(self) -> str:
        if not self.history:
            return "No conversation history yet."
        lines = []
        for msg in self.history:
            role = msg['role'].upper()
            content = str(msg.get('content', ''))[:80]
            lines.append(f"[{role}]: {content}...")
        return '\n'.join(lines)


    def cost_summary(self) -> str:
        # groq pricing for llama 3
        input_cost  = (self.total_input_tokens  / 1_000_000) * 0.05
        output_cost = (self.total_output_tokens / 1_000_000) * 0.08
        return (
            f"Input tokens : {self.total_input_tokens}\n"
            f"Output tokens: {self.total_output_tokens}\n"
            f"Total cost   : ${input_cost + output_cost:.6f}"
        )



In [ ]:
a = [2,3,5,6,7]

In [ ]:
a[2:]

[5, 6, 7]

In [ ]:
a

[6, 7]

In [ ]:
c1 = ChatBot(provider="groq")

In [ ]:
msg1 = [ {"role": "user", "content": "weather in delhi/ give answer in 3 lines at max."}]

In [ ]:
a = c1._call_llm(msg1)

In [ ]:
a2 = c1._extract_reply_and_tool_calls(a)

In [ ]:
a2

(None,
 [ChatCompletionMessageFunctionToolCall(id='pm0hvmwc6', function=Function(arguments='{"city":"delhi"}', name='get_weather_data'), type='function')])

In [ ]:
a2[1]

[ChatCompletionMessageFunctionToolCall(id='pm0hvmwc6', function=Function(arguments='{"city":"delhi"}', name='get_weather_data'), type='function')]

In [ ]:
a2[1][0].function.arguments

'{"city":"delhi"}'

In [ ]:
json.loads(a2[1][0].function.arguments)

{'city': 'delhi'}

In [ ]:
TOOLS[a2[1][0].function.name]

<function __main__.get_weather(city: str) -> str>

In [ ]:
a.usage.prompt_tokens

43

In [ ]:
a

ChatCompletion(id='chatcmpl-42431e73-d75e-484d-ace7-5931d16fd0f9', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='Anime is a style of Japanese animation that has gained worldwide popularity for its vibrant visuals, engaging storylines, and memorable characters. It encompasses a wide range of genres, from action and adventure to romance and comedy, and often incorporates elements of Japanese culture and history. From classic series like "Dragon Ball" and "Sailor Moon" to modern hits like "Attack on Titan" and "Your Lie in April," anime has become a beloved and diverse form of entertainment.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774429983, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_d317489708', usage=CompletionUsage(completion_tokens=95, prompt_tokens=43, total_tokens=138, completion_tokens_details=N

In [ ]:
a.choices[0].message.tool_calls

In [ ]:
llm_reply = a.choices[0].message.content
tool_calling_name = a.choices[0].message.tool_calls

In [ ]:
c2 = ChatBot(provider="anthropic")

In [ ]:
b = c2._call_llm(msg1)

In [ ]:
h = c1._call_llm(msg1)

TypeError: Object of type function is not JSON serializable

In [ ]:
h


ChatCompletion(id='chatcmpl-19cabe25-b238-447b-b15f-39bf18a013cd', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="Delhi's weather is generally hot and humid during summer (April to June) with temperatures ranging from 38°C to 45°C. In winters (December to February), it's relatively cool with temperatures between 2°C to 15°C. The monsoon season in Delhi typically occurs from July to September, bringing moderate rainfall.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1774469272, model='llama-3.1-8b-instant', object='chat.completion', service_tier='on_demand', system_fingerprint='fp_d9492c3c54', usage=CompletionUsage(completion_tokens=68, prompt_tokens=49, total_tokens=117, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.137078943, prompt_time=0.002628052, completion_time=0.100921225, total_time=0.103549277), usage_breakdown=None, x_groq={'id': '

In [ ]:
a = c1._extract_reply_and_tool_calls(h)

In [ ]:
a

("Delhi's weather is generally hot and humid during summer (April to June) with temperatures ranging from 38°C to 45°C. In winters (December to February), it's relatively cool with temperatures between 2°C to 15°C. The monsoon season in Delhi typically occurs from July to September, bringing moderate rainfall.",
 None)

In [1]:
def run_chat(provider:str = 'groq'):
    bot = ChatBot(
        provider = provider,
        system_prompt = "You are a helpful assistant with access about weather, calculator and datetime tool"
    )

    print(f"Chatbot ready (provider: {provider}). Type /help for commands.\n")

    while True:
        user = input("You: ").strip()
        if not user:
            continue
        if user == "/quit":
            print(bot.cost_summary())
            break
        if user == "/help":
            print(bot.help())
            continue
        if user == "/history":
            print(bot.history_summary())
            continue
        if user == "/cost":
            print(bot.cost_summary())
            continue

        print("Assistant: ", end="")
        reply = bot.chat(user)
        print(reply)
        print()

In [ ]:
run_chat(provider="groq")